## 🎯 Learning Outcomes
### By the end of this module, you will be able to

- Understand the three foundational chapters of Python object-oriented programming:
  - **1 Classes & Objects** — defining classes, constructors, variables, namespaces, identity, and representation
  - **2 Attributes & Methods** — instance/class/static methods, properties, access conventions, slots
  - **3 Inheritance & Polymorphism** — single/multiple inheritance, MRO, ABCs, duck typing, mixins

---

## Part 1 — Classes & Objects

### 1.1 The `class` Keyword — Defining a Class and Instantiating Objects

A **class** is a blueprint. An **object** (instance) is a concrete realisation of that blueprint.
Python creates a new namespace for each class and each instance.

In [1]:
# Minimal class — body can be just a docstring or 'pass'
class Dog:
    """A simple Dog class."""

# Instantiation: calling the class like a function creates an object
fido  = Dog()
buddy = Dog()

print(type(fido))          # <class '__main__.Dog'>
print(isinstance(fido, Dog))  # True
print(fido is buddy)       # False — two distinct objects

<class '__main__.Dog'>
True
False


### 1.2 The `__init__` Constructor and `self`

`__init__` is called automatically after the object is created. `self` is a reference to the **new instance** — Python passes it implicitly when you call a method on an object.

In [2]:
class Dog:
    """Dog with name and breed."""

    def __init__(self, name: str, breed: str, age: int = 0):
        # self.x = ... creates an INSTANCE variable
        self.name  = name
        self.breed = breed
        self.age   = age

    def bark(self):
        # 'self' gives access to this specific instance
        return f"{self.name} says: Woof!"

fido  = Dog("Fido",  "Labrador", 3)
buddy = Dog("Buddy", "Beagle",   5)

print(fido.bark())
print(buddy.bark())
print(fido.name, fido.breed, fido.age)

Fido says: Woof!
Buddy says: Woof!
Fido Labrador 3


### 1.3 Instance Variables vs. Class Variables

| | **Class variable** | **Instance variable** |
|--|--|--|
| Defined | Inside class body, outside `__init__` | Inside `__init__` (or any method) via `self.x` |
| Shared? | Yes — all instances see the same value | No — each instance has its own copy |
| Access | `ClassName.var` or `instance.var` | `instance.var` (or `self.var` inside methods) |

In [3]:
class Dog:
    species = "Canis lupus familiaris"  # ← class variable (shared)
    count   = 0                          # ← tracks how many dogs exist

    def __init__(self, name: str):
        self.name = name                 # ← instance variable (unique per dog)
        Dog.count += 1                   # mutate via class name to avoid shadowing

d1 = Dog("Rex")
d2 = Dog("Luna")

print(Dog.species)          # accessed on the class
print(d1.species)           # inherited from class (not a copy)
print(Dog.count)            # 2

# ⚠️  Assigning to d1.species creates an INSTANCE attribute that shadows the class var
d1.species = "Mutant Dog"
print(d1.species)           # 'Mutant Dog'  — instance attribute
print(d2.species)           # unchanged — still 'Canis lupus familiaris'
print(Dog.species)          # unchanged

Canis lupus familiaris
Canis lupus familiaris
2
Mutant Dog
Canis lupus familiaris
Canis lupus familiaris


### 1.4 Class and Instance Namespaces

Python uses a chain of dictionaries (`__dict__`) to look up attributes: **instance → class → base classes**.

In [4]:
class Counter:
    step = 1            # class variable

    def __init__(self, start=0):
        self.value = start

c = Counter(10)

print("Instance __dict__:", c.__dict__)       # {'value': 10}
print("Class __dict__ keys:", list(Counter.__dict__.keys()))

# Attribute lookup order: instance dict → class dict → bases
print("c.step  :", c.step)     # found in class dict
print("c.value :", c.value)    # found in instance dict

# vars() is equivalent to __dict__
print("vars(c) :", vars(c))

Instance __dict__: {'value': 10}
Class __dict__ keys: ['__module__', 'step', '__init__', '__dict__', '__weakref__', '__doc__']
c.step  : 1
c.value : 10
vars(c) : {'value': 10}


### 1.5 Object Identity: `id()`, `is`, and `==`

| Operator | Tests | Uses |
|----------|-------|------|
| `==` | **equality** — calls `__eq__` | comparing values |
| `is` | **identity** — same memory address | checking for `None`, singletons |
| `id()` | returns the object's unique integer address | debugging, understanding aliasing |

In [5]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y

p1 = Point(3, 4)
p2 = Point(3, 4)   # same values, different object
p3 = p1            # alias — same object

print(f"id(p1) = {id(p1)}")
print(f"id(p2) = {id(p2)}")
print(f"id(p3) = {id(p3)}")
print()
print(f"p1 == p2 : {p1 == p2}")   # True  — __eq__ compares values
print(f"p1 is p2 : {p1 is p2}")   # False — different objects
print(f"p1 is p3 : {p1 is p3}")   # True  — same object (alias)
print()
# Always use 'is' to check for None
result = None
print(f"result is None     : {result is None}")   # ✅ correct
print(f"result == None     : {result == None}")   # ⚠️ works but not idiomatic

id(p1) = 132839657646528
id(p2) = 132839657931984
id(p3) = 132839657646528

p1 == p2 : True
p1 is p2 : False
p1 is p3 : True

result is None     : True
result == None     : True


### 1.6 Representing Objects: `__str__` vs. `__repr__`

| Method | Called by | Audience | Goal |
|--------|-----------|----------|------|
| `__repr__` | `repr()`, interactive shell | **developers** | unambiguous, ideally eval-able |
| `__str__` | `str()`, `print()` | **users** | readable, human-friendly |

If only `__repr__` is defined, `__str__` falls back to it.

In [6]:
class Vector:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        # Should look like a valid constructor call
        return f"Vector({self.x!r}, {self.y!r})"

    def __str__(self) -> str:
        # Human-readable
        return f"({self.x}, {self.y})"

v = Vector(3.0, 4.0)

print(repr(v))   # Vector(3.0, 4.0)
print(str(v))    # (3.0, 4.0)
print(v)         # calls __str__ via print()
print(f"{v!r}") # forces __repr__
print(f"{v!s}") # forces __str__

# In a list, repr() is used
vectors = [Vector(1, 2), Vector(3, 4)]
print(vectors)

Vector(3.0, 4.0)
(3.0, 4.0)
(3.0, 4.0)
Vector(3.0, 4.0)
(3.0, 4.0)
[Vector(1, 2), Vector(3, 4)]


---

## Part 2 — Attributes & Methods

### 2.1 Instance Methods, Class Methods, and Static Methods

In [7]:
from datetime import date

class Employee:
    company = "Acme Corp"        # class variable

    def __init__(self, name: str, salary: float):
        self.name   = name
        self.salary = salary

    # ── Instance method ───────────────────────────────────────────────────────
    # First argument is 'self' — the specific instance
    def give_raise(self, amount: float) -> None:
        """Modify this employee's salary."""
        self.salary += amount

    # ── Class method ──────────────────────────────────────────────────────────
    # First argument is 'cls' — the class itself (not an instance)
    # Commonly used as alternative constructors
    @classmethod
    def from_string(cls, emp_str: str) -> "Employee":
        """Alternative constructor: parse 'Name-Salary' string."""
        name, salary = emp_str.split("-")
        return cls(name.strip(), float(salary.strip()))

    @classmethod
    def get_company(cls) -> str:
        return cls.company

    # ── Static method ─────────────────────────────────────────────────────────
    # No implicit first argument — just a plain function scoped to the class
    # Use when logic belongs conceptually to the class but needs no class/instance state
    @staticmethod
    def is_workday(day: date) -> bool:
        """Return True if day is Monday–Friday."""
        return day.weekday() < 5

    def __repr__(self):
        return f"Employee({self.name!r}, {self.salary})"

# Usage
e1 = Employee("Alice", 70_000)
e2 = Employee.from_string("Bob - 80000")   # class method as factory

e1.give_raise(5_000)                        # instance method
print(e1, e2)

print(Employee.get_company())               # class method on class
print(e1.get_company())                     # class method on instance (also works)

today = date.today()
print(f"{today} is a workday: {Employee.is_workday(today)}")  # static method

Employee('Alice', 75000) Employee('Bob', 80000.0)
Acme Corp
Acme Corp
2026-05-17 is a workday: False


### 2.2 Properties — `@property` and the Getter/Setter Pattern

Properties let you expose **attribute-style access** while running validation or computation behind the scenes. No parentheses needed at the call site.

In [8]:
class Temperature:
    """Stores temperature in Celsius, exposes Fahrenheit as a property."""

    def __init__(self, celsius: float = 0.0):
        self.celsius = celsius   # uses the setter below

    # ── getter ────────────────────────────────────────────────────────────────
    @property
    def celsius(self) -> float:
        return self._celsius

    # ── setter ────────────────────────────────────────────────────────────────
    @celsius.setter
    def celsius(self, value: float) -> None:
        if value < -273.15:
            raise ValueError(f"Temperature {value}°C is below absolute zero")
        self._celsius = value

    # ── deleter (optional) ────────────────────────────────────────────────────
    @celsius.deleter
    def celsius(self):
        del self._celsius

    # ── computed read-only property ───────────────────────────────────────────
    @property
    def fahrenheit(self) -> float:
        """Read-only — no setter defined."""
        return self._celsius * 9 / 5 + 32

    def __repr__(self):
        return f"Temperature({self._celsius}°C / {self.fahrenheit:.1f}°F)"

t = Temperature(100)
print(t)
t.celsius = 0
print(t)

# Setter validation
try:
    t.celsius = -300
except ValueError as e:
    print(f"ValueError: {e}")

# Read-only: setting fahrenheit raises AttributeError
try:
    t.fahrenheit = 100
except AttributeError as e:
    print(f"AttributeError: {e}")

Temperature(100°C / 212.0°F)
Temperature(0°C / 32.0°F)
ValueError: Temperature -300°C is below absolute zero
AttributeError: property 'fahrenheit' of 'Temperature' object has no setter


### 2.3 Private and Protected Conventions — `_` and `__` Name Mangling

In [9]:
class BankAccount:
    def __init__(self, owner: str, balance: float):
        self.owner       = owner      # ← public: anyone can read/write
        self._balance    = balance    # ← protected: convention only — "internal use"
        self.__pin       = "1234"     # ← private: name-mangled by Python

    def deposit(self, amount: float):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount

    def verify_pin(self, pin: str) -> bool:
        return self.__pin == pin      # access within class works fine

    def __repr__(self):
        return f"BankAccount({self.owner!r}, £{self._balance:.2f})"

acc = BankAccount("Alice", 1000)
print(acc)

# Public — fine
print(acc.owner)

# Protected — accessible but signals "don't touch"
print(acc._balance)

# Private — name is mangled to _BankAccount__pin
try:
    print(acc.__pin)           # AttributeError
except AttributeError as e:
    print(f"AttributeError: {e}")

# Name-mangled name is still reachable (not truly private)
print("Mangled access:", acc._BankAccount__pin)

print("PIN OK:", acc.verify_pin("1234"))

BankAccount('Alice', £1000.00)
Alice
1000
AttributeError: 'BankAccount' object has no attribute '__pin'
Mangled access: 1234
PIN OK: True


In [10]:
# Name mangling protects double-underscore attributes from accidental
# override in subclasses
class Child(BankAccount):
    def __init__(self, owner, balance):
        super().__init__(owner, balance)
        self.__pin = "9999"   # becomes _Child__pin — does NOT override _BankAccount__pin

c = Child("Bob", 500)
print("Child PIN   :", c._Child__pin)       # 9999
print("Parent PIN  :", c._BankAccount__pin) # 1234 — untouched

Child PIN   : 9999
Parent PIN  : 1234


---

## Part 3 — Inheritance & Polymorphism

### 3.1 Single Inheritance — Parent, Child, and `super()`

In [11]:
class Animal:
    """Base class."""

    def __init__(self, name: str, sound: str):
        self.name  = name
        self.sound = sound

    def speak(self) -> str:
        return f"{self.name} says {self.sound}!"

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"


class Dog(Animal):             # Dog inherits from Animal
    def __init__(self, name: str, breed: str):
        super().__init__(name, "Woof")   # delegate to parent __init__
        self.breed = breed

    def fetch(self, item: str) -> str:
        return f"{self.name} fetches the {item}!"


class Cat(Animal):
    def __init__(self, name: str, indoor: bool = True):
        super().__init__(name, "Meow")
        self.indoor = indoor

    def speak(self) -> str:    # method override
        base = super().speak()
        return base + " (softly)"


d = Dog("Rex", "German Shepherd")
c = Cat("Whiskers")

print(d.speak())       # inherited from Animal
print(d.fetch("ball"))  # Dog-specific
print(c.speak())       # overridden in Cat

Rex says Woof!
Rex fetches the ball!
Whiskers says Meow! (softly)


### 3.2 Method Resolution Order (MRO) — C3 Linearization

Python uses **C3 linearization** to determine the order in which base classes are searched for a method. `ClassName.__mro__` shows the full chain.

In [12]:
class A:
    def greet(self): return "Hello from A"

class B(A):
    def greet(self): return "Hello from B"

class C(A):
    def greet(self): return "Hello from C"

class D(B, C):   # multiple inheritance
    pass

# MRO: D → B → C → A → object
print("MRO:")
for cls in D.__mro__:
    print(f"  {cls.__name__}")

d = D()
print(d.greet())   # 'Hello from B' — B comes before C in MRO

MRO:
  D
  B
  C
  A
  object
Hello from B


### 3.3 Multiple Inheritance and the Diamond Problem

In [13]:
#         Base
#        /    \
#      Left   Right
#        \    /
#        Diamond

class Base:
    def method(self):
        print("Base.method")

class Left(Base):
    def method(self):
        print("Left.method")
        super().method()   # super() follows MRO, not class hierarchy

class Right(Base):
    def method(self):
        print("Right.method")
        super().method()

class Diamond(Left, Right):
    def method(self):
        print("Diamond.method")
        super().method()

print("MRO:", [c.__name__ for c in Diamond.__mro__])
print()
Diamond().method()
# Diamond → Left → Right → Base  (each Base.method called once — cooperative inheritance)

MRO: ['Diamond', 'Left', 'Right', 'Base', 'object']

Diamond.method
Left.method
Right.method
Base.method


### 3.4 Abstract Base Classes — `abc` module and `@abstractmethod`

ABCs define an **interface** that subclasses must implement. Instantiating a class with unimplemented abstract methods raises `TypeError`.

In [14]:
from abc import ABC, abstractmethod

class Shape(ABC):         # inheriting from ABC marks this as abstract
    """Abstract shape — cannot be instantiated directly."""

    @abstractmethod
    def area(self) -> float:
        """Return the area of the shape."""

    @abstractmethod
    def perimeter(self) -> float:
        """Return the perimeter of the shape."""

    def describe(self) -> str:    # concrete method — available to all subclasses
        return f"{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"


class Circle(Shape):
    import math as _math
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        import math
        return math.pi * self.radius ** 2

    def perimeter(self) -> float:
        import math
        return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, w: float, h: float):
        self.w, self.h = w, h

    def area(self) -> float:      return self.w * self.h
    def perimeter(self) -> float: return 2 * (self.w + self.h)


# Cannot instantiate abstract class
try:
    s = Shape()
except TypeError as e:
    print(f"TypeError: {e}")

shapes = [Circle(5), Rectangle(4, 6)]
for shape in shapes:
    print(shape.describe())

TypeError: Can't instantiate abstract class Shape without an implementation for abstract methods 'area', 'perimeter'
Circle: area=78.54, perimeter=31.42
Rectangle: area=24.00, perimeter=20.00


### 3.5 Polymorphism — Duck Typing and Interface-Based Design

> *"If it walks like a duck and quacks like a duck, it's a duck."*

Python checks for **behaviour** (method/attribute presence), not type. You don't need a shared base class — just a compatible interface.

In [15]:
# No common base class needed — duck typing in action
class Dog:
    def speak(self): return "Woof!"

class Cat:
    def speak(self): return "Meow!"

class Robot:
    def speak(self): return "Beep boop."

def make_noise(entity):     # works with anything that has .speak()
    print(entity.speak())

for creature in [Dog(), Cat(), Robot()]:
    make_noise(creature)

Woof!
Meow!
Beep boop.


In [16]:
# Interface-based design — use ABCs to make the contract explicit
# while still allowing polymorphic use
from abc import ABC, abstractmethod

class Serializable(ABC):
    @abstractmethod
    def to_dict(self) -> dict: ...

    @abstractmethod
    def from_dict(cls, data: dict): ...

class User(Serializable):
    def __init__(self, name, email):
        self.name, self.email = name, email

    def to_dict(self):
        return {"name": self.name, "email": self.email}

    @classmethod
    def from_dict(cls, data):
        return cls(data["name"], data["email"])

class Product(Serializable):
    def __init__(self, sku, price):
        self.sku, self.price = sku, price

    def to_dict(self):
        return {"sku": self.sku, "price": self.price}

    @classmethod
    def from_dict(cls, data):
        return cls(data["sku"], data["price"])

def save_all(objects: list) -> list:
    """Polymorphic — works for any Serializable."""
    return [obj.to_dict() for obj in objects]

items = [User("Alice", "alice@ex.com"), Product("P001", 29.99)]
print(save_all(items))

[{'name': 'Alice', 'email': 'alice@ex.com'}, {'sku': 'P001', 'price': 29.99}]


### 3.6 Mixins — Compositional Pattern

A **mixin** is a class that provides reusable behaviour to be combined via multiple inheritance. It is not meant to be instantiated on its own. Heavy use in **Keras** (`Layer`, `Model`) and **Django** (`LoginRequiredMixin`, `PermissionRequiredMixin`).

In [17]:
import json
import datetime

# ── Mixin classes — each adds one orthogonal capability ───────────────────────

class JsonMixin:
    """Adds JSON serialisation to any class with a to_dict() method."""
    def to_json(self) -> str:
        return json.dumps(self.to_dict(), indent=2)

    @classmethod
    def from_json(cls, json_str: str):
        return cls.from_dict(json.loads(json_str))


class TimestampMixin:
    """Adds created_at / updated_at tracking."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)      # cooperative — passes args along
        self.created_at = datetime.datetime.now().isoformat()
        self.updated_at = self.created_at

    def touch(self):
        self.updated_at = datetime.datetime.now().isoformat()


class ValidationMixin:
    """Adds field validation via a validate() hook."""
    def save(self):
        self.validate()           # subclass must define validate()
        print(f"{self!r} saved successfully")


# ── Compose mixins with a domain class ────────────────────────────────────────

class Article(TimestampMixin, JsonMixin, ValidationMixin):
    def __init__(self, title: str, body: str):
        super().__init__()        # TimestampMixin.__init__ runs via MRO
        self.title = title
        self.body  = body

    def to_dict(self):
        return {"title": self.title, "body": self.body,
                "created_at": self.created_at}

    @classmethod
    def from_dict(cls, data):
        return cls(data["title"], data["body"])

    def validate(self):
        if not self.title:
            raise ValueError("Article title cannot be empty")
        if len(self.body) < 10:
            raise ValueError("Article body too short")

    def __repr__(self):
        return f"Article({self.title!r})"


# Usage
a = Article("Python OOP", "Python supports multiple inheritance via mixins.")
print(a.to_json())
a.save()

print("\nMRO:", [c.__name__ for c in Article.__mro__])

{
  "title": "Python OOP",
  "body": "Python supports multiple inheritance via mixins.",
  "created_at": "2026-05-17T03:34:01.652641"
}
Article('Python OOP') saved successfully

MRO: ['Article', 'TimestampMixin', 'JsonMixin', 'ValidationMixin', 'object']


### 3.7 `isinstance()` and `issubclass()` for Type Checking

In [18]:
from abc import ABC, abstractmethod

class Vehicle(ABC):
    @abstractmethod
    def move(self): ...

class Car(Vehicle):
    def move(self): return "driving"

class Boat(Vehicle):
    def move(self): return "sailing"

class AmphibiousCar(Car, Boat):
    def move(self): return "driving or sailing"

car  = Car()
boat = Boat()
amph = AmphibiousCar()

# isinstance(obj, cls) — True if obj is an instance of cls OR any subclass
print("isinstance checks:")
print(f"  isinstance(car,  Vehicle)  : {isinstance(car,  Vehicle)}")
print(f"  isinstance(amph, Car)      : {isinstance(amph, Car)}")
print(f"  isinstance(amph, Boat)     : {isinstance(amph, Boat)}")
print(f"  isinstance(car,  Boat)     : {isinstance(car,  Boat)}")

# isinstance with a tuple — checks against multiple types at once
print(f"  isinstance(42, (int, float)) : {isinstance(42, (int, float))}")

# issubclass(sub, sup) — checks class hierarchy (not instances)
print("\nissubclass checks:")
print(f"  issubclass(Car,          Vehicle) : {issubclass(Car, Vehicle)}")
print(f"  issubclass(AmphibiousCar, Boat)   : {issubclass(AmphibiousCar, Boat)}")
print(f"  issubclass(Car,          Boat)    : {issubclass(Car, Boat)}")
print(f"  issubclass(Vehicle,      Vehicle) : {issubclass(Vehicle, Vehicle)}")  # True: a class is a subclass of itself

# Prefer isinstance over type() == for polymorphism
print("\ntype() vs isinstance():")
print(f"  type(amph) is Car       : {type(amph) is Car}")
print(f"  isinstance(amph, Car)   : {isinstance(amph, Car)}")

isinstance checks:
  isinstance(car,  Vehicle)  : True
  isinstance(amph, Car)      : True
  isinstance(amph, Boat)     : True
  isinstance(car,  Boat)     : False
  isinstance(42, (int, float)) : True

issubclass checks:
  issubclass(Car,          Vehicle) : True
  issubclass(AmphibiousCar, Boat)   : True
  issubclass(Car,          Boat)    : False
  issubclass(Vehicle,      Vehicle) : True

type() vs isinstance():
  type(amph) is Car       : False
  isinstance(amph, Car)   : True


## Summary

| Concept | Key Points |
|---|---|
| Define class | `class Foo:` |
| Constructor | `def __init__(self, ...)` |
| Instance variable | `self.x = value` |
| Class variable | `x = value` inside class body |
| Readable repr | `def __repr__(self): return f"Foo(...)"` |
| User-friendly str | `def __str__(self): return "..."` |
| Identity check | `a is b`, `id(a)` |
| Equality check | `a == b` (calls `__eq__`) |
| Instance method | `def method(self):` |
| Class method | `@classmethod` / `def method(cls):` |
| Static method | `@staticmethod` / `def method():` |
| Property getter | `@property` |
| Property setter | `@prop.setter` |
| Protected attr | `self._name` (convention) |
| Private attr | `self.__name` (name-mangled) |
| Memory opt. | `__slots__ = ("x", "y")` |
| Inherit | `class Child(Parent):` |
| Call parent | `super().__init__(...)` |
| View MRO | `ClassName.__mro__` |
| Abstract class | `class Foo(ABC):` + `@abstractmethod` |
| Type check | `isinstance(obj, Cls)` |
| Subclass check | `issubclass(Sub, Sup)` |
| Mixin pattern | `class Child(MixinA, MixinB, Base):` |

### Contributed by: Abdulhadi Zubailah